# 14: Knowledge Graph Analytics

Graph-theoretic structure analysis (NetworkX) and KG-native multi-hop inference (SPARQL).

**Prerequisite**: notebook 12 (materialization of SR-05/06/10) and notebook 13 (OWL classification) must have run.

---

## Contents

1. [Graph-Theoretic Structure](#part-1--graph-theoretic-structure) — centrality, communities, flow, trajectory similarity
2. [Multi-Hop Inference Chains](#part-2--multi-hop-inference-chains) — co-visitation reach, flow chains, user-to-POI paths
3. [Ontological Cross-Tabulations](#part-3--ontological-cross-tabulations) — purpose×interaction, engagement×category, chi-squared tests

In [ ]:
import sys
sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns

from helpers.graph_builder import TrajectoNXGraphBuilder
from helpers.fuseki_config import query_endpoint
from helpers.fuseki_queries import execute_sparql_query, format_results

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
%matplotlib inline

# NetworkX graph builder
builder = TrajectoNXGraphBuilder()

# SPARQL helper
PREFIXES = """
PREFIX trajecto: <http://w3id.org/trajecto#>
PREFIX poi:      <http://w3id.org/trajecto/poi#>
PREFIX patterns: <http://w3id.org/trajecto/patterns#>
PREFIX context:  <http://w3id.org/trajecto/context#>
PREFIX user:     <http://w3id.org/trajecto/user#>
PREFIX purpose:  <http://w3id.org/trajecto/purpose#>
PREFIX events:   <http://w3id.org/trajecto/events#>
PREFIX dul:      <http://www.ontologydesignpatterns.org/ont/dul/DUL.owl#>
PREFIX rdf:      <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs:     <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd:      <http://www.w3.org/2001/XMLSchema#>
"""

def query(sparql: str) -> pd.DataFrame:
    """Execute SPARQL query and return DataFrame."""
    return format_results(execute_sparql_query(query_endpoint, PREFIXES + sparql))

def short(uri: str) -> str:
    """Shorten a URI to its local name."""
    return uri.rsplit('#', 1)[-1].rsplit('/', 1)[-1]

---

## Part 1 — Graph-Theoretic Structure

Centrality, community detection, flow patterns, and trajectory similarity via NetworkX.
Graphs built live from SPARQL via `graph_builder.py`.

In [ ]:
# Build the graphs we need
print("Building graph structures from knowledge graph...")

# POI co-visitation graph (undirected)
poi_graph = builder.build_poi_covisit_graph()

# POI flow graph (directed - shows transitions)
poi_flow = builder.build_poi_flow_graph()

# Trajectory similarity graph (shared POIs)
traj_sim_graph = builder.build_trajectory_similarity_graph(min_shared=1)

# POI visit sequences per trajectory
poi_sequences = builder.get_trajectory_poi_sequences()

# Load POI labels for readable output
poi_metadata = builder.get_poi_metadata()
poi_labels = {}
if not poi_metadata.empty:
    for _, row in poi_metadata.iterrows():
        poi_uri = row['poi']
        label = row.get('label')
        if isinstance(label, str) and label and poi_uri not in poi_labels:
            poi_labels[poi_uri] = label
    print(f"Loaded labels for {len(poi_labels)} POIs")


def poi_name(uri):
    """Get a readable name for a POI: label if available, otherwise short ID."""
    if not isinstance(uri, str):
        return str(uri)
    label = poi_labels.get(uri)
    short_id = uri.split('/')[-1].split('#')[-1]
    if isinstance(label, str) and label:
        return f"{label[:25]} ({short_id[-10:]})"
    return short_id[:35]


print(f"\nPOI Co-visitation Graph: {poi_graph.number_of_nodes()} nodes, {poi_graph.number_of_edges()} edges")
print(f"POI Flow Graph: {poi_flow.number_of_nodes()} nodes, {poi_flow.number_of_edges()} edges")
print(f"Trajectory Similarity Graph: {traj_sim_graph.number_of_nodes()} nodes, {traj_sim_graph.number_of_edges()} edges")
print(f"POI Sequences: {len(poi_sequences)} trajectories")


---

### 1.1 POI Importance Analysis

We measure POI importance using three intuitive metrics:

| Metric | Question It Answers | Real-World Meaning |
|--------|---------------------|--------------------|
| **Popularity Score** | How many visitors? | Tourist hotspots |
| **Bridge Score** | Does it connect different areas? | Transit hubs, crossroads |
| **Influence Score** | How "important" in the network? | Must-see destinations |

In [ ]:
# Calculate importance metrics
if poi_graph.number_of_nodes() > 0:
    print("Calculating POI importance metrics...")
    print("="*60)

    # 1. POPULARITY SCORE (Degree Centrality)
    # "How many other POIs is this one visited with?"
    degree_centrality = nx.degree_centrality(poi_graph)

    # 2. BRIDGE SCORE (Betweenness Centrality)
    # "How often is this POI on the shortest path between other POIs?"
    betweenness_centrality = nx.betweenness_centrality(poi_graph)

    # 3. INFLUENCE SCORE (PageRank)
    # "How important is this POI based on connections to other important POIs?"
    pagerank = nx.pagerank(poi_graph)

    print("Metrics calculated successfully!\n")

    # Build results dataframe
    poi_metrics = pd.DataFrame({
        'poi': list(poi_graph.nodes()),
        'popularity_score': [degree_centrality[n] for n in poi_graph.nodes()],
        'bridge_score': [betweenness_centrality[n] for n in poi_graph.nodes()],
        'influence_score': [pagerank[n] for n in poi_graph.nodes()]
    })

    # Use poi_name() for readable labels
    poi_metrics['name'] = poi_metrics['poi'].apply(poi_name)

    print(f"Analyzed {len(poi_metrics)} POIs")
else:
    print("No POI data available for analysis.")
    poi_metrics = pd.DataFrame()

In [ ]:
# Display top POIs by each metric
if not poi_metrics.empty:
    print("\n" + "="*80)
    print("TOP 10 POIs BY POPULARITY SCORE")
    print("(POIs visited with the most other POIs - tourist hotspots)")
    print("="*80)
    top_popular = poi_metrics.nlargest(10, 'popularity_score')[['name', 'popularity_score']]
    for i, (_, row) in enumerate(top_popular.iterrows(), 1):
        bar = '█' * int(row['popularity_score'] * 40)
        print(f"{i:2d}. {row['name']:<40} {row['popularity_score']:.4f} {bar}")

In [ ]:
if not poi_metrics.empty:
    print("\n" + "="*80)
    print("TOP 10 POIs BY BRIDGE SCORE")
    print("(POIs that connect different areas - transit hubs)")
    print("="*80)
    top_bridges = poi_metrics.nlargest(10, 'bridge_score')[['name', 'bridge_score']]
    for i, (_, row) in enumerate(top_bridges.iterrows(), 1):
        bar = '█' * int(row['bridge_score'] * 100)
        print(f"{i:2d}. {row['name']:<40} {row['bridge_score']:.4f} {bar}")

In [ ]:
if not poi_metrics.empty:
    print("\n" + "="*80)
    print("TOP 10 POIs BY INFLUENCE SCORE")
    print("(Most influential POIs - must-see destinations)")
    print("="*80)
    top_influential = poi_metrics.nlargest(10, 'influence_score')[['name', 'influence_score']]
    for i, (_, row) in enumerate(top_influential.iterrows(), 1):
        bar = '█' * int(row['influence_score'] * 200)
        print(f"{i:2d}. {row['name']:<40} {row['influence_score']:.4f} {bar}")

In [ ]:
# Visualize metric correlations
if not poi_metrics.empty and len(poi_metrics) > 5:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Popularity vs Bridge
    ax1 = axes[0]
    ax1.scatter(poi_metrics['popularity_score'], poi_metrics['bridge_score'],
                alpha=0.6, c='#3498db', s=50)
    ax1.set_xlabel('Popularity Score', fontsize=12)
    ax1.set_ylabel('Bridge Score', fontsize=12)
    ax1.set_title('Popularity vs Bridge\n(Top-right = popular transit hubs)', fontsize=12, fontweight='bold')

    # Popularity vs Influence
    ax2 = axes[1]
    ax2.scatter(poi_metrics['popularity_score'], poi_metrics['influence_score'],
                alpha=0.6, c='#e74c3c', s=50)
    ax2.set_xlabel('Popularity Score', fontsize=12)
    ax2.set_ylabel('Influence Score', fontsize=12)
    ax2.set_title('Popularity vs Influence\n(Correlation shows importance)', fontsize=12, fontweight='bold')

    # Bridge vs Influence
    ax3 = axes[2]
    ax3.scatter(poi_metrics['bridge_score'], poi_metrics['influence_score'],
                alpha=0.6, c='#2ecc71', s=50)
    ax3.set_xlabel('Bridge Score', fontsize=12)
    ax3.set_ylabel('Influence Score', fontsize=12)
    ax3.set_title('Bridge vs Influence\n(Connectors vs Destinations)', fontsize=12, fontweight='bold')

    plt.tight_layout()
    plt.show()

    print("\nInterpretation:")
    print("  - High Popularity + High Bridge = Major hub (e.g., central station)")
    print("  - High Popularity + Low Bridge = Destination (e.g., famous museum)")
    print("  - Low Popularity + High Bridge = Hidden connector (niche shortcut)")

---

### 1.2 Community Detection

**Key Question**: Which POIs form natural clusters?

Communities are groups of POIs that are more connected to each other than to POIs outside the group. This can reveal:
- Geographic neighborhoods
- Thematic clusters (cultural areas, food districts)
- Natural tour routes

In [ ]:
# Detect communities using Louvain algorithm
if poi_graph.number_of_nodes() > 0:
    print("Detecting POI communities...")
    print("="*60)

    try:
        # Try networkx >= 2.7 first
        from networkx.algorithms.community import louvain_communities
        communities_result = louvain_communities(poi_graph, seed=42)
        communities = [list(c) for c in communities_result]
    except ImportError:
        # Fall back to python-louvain package for networkx < 2.7
        try:
            import community.community_louvain as community_louvain
            partition = community_louvain.best_partition(poi_graph, random_state=42)

            # Convert partition dict to list of communities
            community_to_nodes = {}
            for node, comm_id in partition.items():
                if comm_id not in community_to_nodes:
                    community_to_nodes[comm_id] = []
                community_to_nodes[comm_id].append(node)
            communities = list(community_to_nodes.values())
        except ImportError:
            print("Community detection requires either:")
            print("  - networkx >= 2.7, OR")
            print("  - python-louvain package (pip install python-louvain)")
            communities = []

    if communities:
        print(f"\nFound {len(communities)} communities\n")

        # Analyze each community
        print(f"{'Community':<12} {'Size':<8} {'Sample POIs'}")
        print("-"*80)

        for i, community in enumerate(sorted(communities, key=len, reverse=True)[:10], 1):
            sample_pois = [poi_name(p)[:25] for p in community[:3]]
            sample_str = ', '.join(sample_pois)
            if len(community) > 3:
                sample_str += f' (+{len(community)-3} more)'
            print(f"Community {i:<3} {len(community):<8} {sample_str}")

        # Create community mapping
        poi_to_community = {}
        for i, community in enumerate(communities):
            for poi in community:
                poi_to_community[poi] = i
    else:
        poi_to_community = {}
else:
    print("No POI data available for community detection.")
    communities = []
    poi_to_community = {}

In [ ]:
# Visualize communities
if communities and poi_graph.number_of_nodes() > 0:
    # Community size distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Bar chart of community sizes
    ax1 = axes[0]
    sizes = sorted([len(c) for c in communities], reverse=True)
    colors = plt.cm.Set3(np.linspace(0, 1, len(sizes)))
    ax1.bar(range(1, len(sizes)+1), sizes, color=colors, edgecolor='white')
    ax1.set_xlabel('Community Rank', fontsize=12)
    ax1.set_ylabel('Number of POIs', fontsize=12)
    ax1.set_title('Community Sizes\n(Larger = more POIs clustered together)',
                  fontsize=14, fontweight='bold')

    # Network visualization with communities colored
    ax2 = axes[1]

    # Use only largest connected component for visualization
    if nx.is_connected(poi_graph):
        G_viz = poi_graph
    else:
        largest_cc = max(nx.connected_components(poi_graph), key=len)
        G_viz = poi_graph.subgraph(largest_cc).copy()

    # Limit nodes for readability
    if G_viz.number_of_nodes() > 100:
        top_nodes = sorted(G_viz.degree(), key=lambda x: x[1], reverse=True)[:100]
        top_node_set = {n for n, d in top_nodes}
        G_viz = G_viz.subgraph(top_node_set).copy()

    if G_viz.number_of_nodes() > 2:
        pos = nx.spring_layout(G_viz, k=1.5, iterations=50, seed=42)

        # Color by community
        node_colors = [poi_to_community.get(n, 0) for n in G_viz.nodes()]

        nx.draw_networkx_edges(G_viz, pos, alpha=0.2, ax=ax2)
        nx.draw_networkx_nodes(G_viz, pos, node_color=node_colors,
                               cmap=plt.cm.Set3, node_size=100, alpha=0.8, ax=ax2)

        ax2.set_title(f'POI Network by Community\n({G_viz.number_of_nodes()} POIs shown)',
                      fontsize=14, fontweight='bold')
        ax2.axis('off')

    plt.tight_layout()
    plt.show()

    print("\nInsight: Each color represents a cluster of frequently co-visited POIs.")
    print("These often correspond to geographic areas or thematic groupings.")

---

### 1.3 Path Analysis

**Key Question**: What are the common routes between POIs?

The POI flow graph shows directional transitions: "After visiting POI A, travelers often go to POI B."

In [ ]:
# Analyze POI transitions
if poi_flow.number_of_edges() > 0:
    print("POI Flow Analysis")
    print("="*60)
    print(f"\nTotal transitions tracked: {poi_flow.number_of_edges()}")

    # Find most common transitions
    transitions = [(u, v, d.get('weight', 1))
                   for u, v, d in poi_flow.edges(data=True)]
    transitions.sort(key=lambda x: x[2], reverse=True)

    print("\nTop 15 Most Common POI Transitions:")
    print("-"*85)
    print(f"{'From POI':<35} {'To POI':<35} {'Count':<10}")
    print("-"*85)

    for poi1, poi2, count in transitions[:15]:
        name1 = poi_name(poi1)[:33]
        name2 = poi_name(poi2)[:33]
        print(f"{name1:<35} -> {name2:<35} {count:<10}")
else:
    print("No POI flow data available.")

In [ ]:
# Find articulation points (critical connector POIs)
if poi_graph.number_of_nodes() > 0 and nx.is_connected(poi_graph):
    articulation_points = list(nx.articulation_points(poi_graph))

    print("\nCritical Connector POIs (Articulation Points):")
    print("="*60)
    print("These POIs, if removed, would disconnect parts of the network.")
    print("They are critical for maintaining route connectivity.\n")

    if articulation_points:
        for poi in articulation_points[:10]:
            name = poi_name(poi)
            degree = poi_graph.degree(poi)
            print(f"  - {name} (connections: {degree})")

        if len(articulation_points) > 10:
            print(f"  ... and {len(articulation_points)-10} more")
    else:
        print("  No critical connector POIs found (network is well-connected).")
elif poi_graph.number_of_nodes() > 0:
    print("\nNetwork is not fully connected - analyzing largest component...")
    largest_cc = max(nx.connected_components(poi_graph), key=len)
    G_largest = poi_graph.subgraph(largest_cc).copy()

    articulation_points = list(nx.articulation_points(G_largest))
    print(f"Found {len(articulation_points)} critical connectors in the main network.")

In [ ]:
# Analyze out-degree (POIs people leave from)
if poi_flow.number_of_nodes() > 0:
    out_degrees = dict(poi_flow.out_degree())
    in_degrees = dict(poi_flow.in_degree())

    print("\nPOI Role Analysis (based on flow patterns):")
    print("="*60)

    # Classify POIs by flow pattern
    starting_points = []  # High out, low in
    destinations = []      # High in, low out
    hubs = []             # High both

    for poi in poi_flow.nodes():
        out_d = out_degrees.get(poi, 0)
        in_d = in_degrees.get(poi, 0)

        if out_d > 3 and in_d > 3:
            hubs.append((poi, out_d + in_d))
        elif out_d > in_d * 2 and out_d > 2:
            starting_points.append((poi, out_d))
        elif in_d > out_d * 2 and in_d > 2:
            destinations.append((poi, in_d))

    print("\nStarting Points (where journeys begin):")
    for poi, _count in sorted(starting_points, key=lambda x: x[1], reverse=True)[:5]:
        print(f"  - {poi_name(poi)}")

    print("\nDestinations (where journeys end):")
    for poi, _count in sorted(destinations, key=lambda x: x[1], reverse=True)[:5]:
        print(f"  - {poi_name(poi)}")

    print("\nMajor Hubs (high traffic both directions):")
    for poi, _count in sorted(hubs, key=lambda x: x[1], reverse=True)[:5]:
        print(f"  - {poi_name(poi)}")

In [ ]:
# Trajectory similarity and sequence length analysis
print("Trajectory & Sequence Analysis")
print("=" * 60)

# Trajectory similarity graph stats
if traj_sim_graph.number_of_nodes() > 0:
    degrees = [d for _, d in traj_sim_graph.degree()]
    print("\nTrajectory Similarity Graph:")
    print(f"  Trajectories: {traj_sim_graph.number_of_nodes()}")
    print(f"  Similarity edges: {traj_sim_graph.number_of_edges()}")
    print(f"  Avg similar trajectories per trajectory: {np.mean(degrees):.1f}")
    print(f"  Max similar trajectories: {max(degrees)}")
    components = list(nx.connected_components(traj_sim_graph))
    print(f"  Connected components: {len(components)}")
    print(f"  Largest component: {max(len(c) for c in components)} trajectories")

# Sequence length distribution
if poi_sequences:
    seq_lengths = [len(seq) for seq in poi_sequences.values()]
    print("\nPOI Sequence Length Distribution:")
    print(f"  Mean: {np.mean(seq_lengths):.1f} POIs")
    print(f"  Median: {np.median(seq_lengths):.1f} POIs")
    print(f"  Min: {min(seq_lengths)}, Max: {max(seq_lengths)}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Sequence length histogram
    ax1 = axes[0]
    ax1.hist(seq_lengths, bins=range(1, max(seq_lengths) + 2), color='#3498db',
             edgecolor='white', alpha=0.8)
    ax1.axvline(np.median(seq_lengths), color='red', linestyle='--', label=f'Median ({np.median(seq_lengths):.0f})')
    ax1.set_xlabel('Number of POIs', fontsize=12)
    ax1.set_ylabel('Number of Trajectories', fontsize=12)
    ax1.set_title('POI Sequence Length Distribution', fontsize=14, fontweight='bold')
    ax1.legend()

    # Trajectory similarity degree distribution
    ax2 = axes[1]
    if degrees:
        ax2.hist(degrees, bins=30, color='#2ecc71', edgecolor='white', alpha=0.8)
        ax2.axvline(np.mean(degrees), color='red', linestyle='--', label=f'Mean ({np.mean(degrees):.1f})')
        ax2.set_xlabel('Number of Similar Trajectories', fontsize=12)
        ax2.set_ylabel('Count', fontsize=12)
        ax2.set_title('Trajectory Similarity Degree Distribution', fontsize=14, fontweight='bold')
        ax2.legend()

    plt.tight_layout()
    plt.show()

---

### 1.4 Summary & Combined Ranking

Let's compile the key findings from our graph analytics.

In [ ]:
# Compile summary
print("\n" + "="*70)
print("GRAPH ANALYTICS SUMMARY")
print("="*70)

if not poi_metrics.empty:
    print("\n1. POI IMPORTANCE RANKINGS")
    print("-"*40)

    print("\n   Most Popular POIs (by co-visitation):")
    for i, (_, row) in enumerate(poi_metrics.nlargest(3, 'popularity_score').iterrows(), 1):
        print(f"   {i}. {row['name']}")

    print("\n   Key Bridge POIs (connecting areas):")
    for i, (_, row) in enumerate(poi_metrics.nlargest(3, 'bridge_score').iterrows(), 1):
        print(f"   {i}. {row['name']}")

    print("\n   Most Influential POIs:")
    for i, (_, row) in enumerate(poi_metrics.nlargest(3, 'influence_score').iterrows(), 1):
        print(f"   {i}. {row['name']}")

if communities:
    print("\n2. COMMUNITY STRUCTURE")
    print("-"*40)
    print(f"   Number of POI communities: {len(communities)}")
    print(f"   Largest community: {max(len(c) for c in communities)} POIs")
    print(f"   Average community size: {np.mean([len(c) for c in communities]):.1f} POIs")

if poi_flow.number_of_edges() > 0:
    print("\n3. FLOW PATTERNS")
    print("-"*40)
    print(f"   Total tracked transitions: {poi_flow.number_of_edges()}")
    if transitions:
        top_trans = transitions[0]
        name1 = poi_name(top_trans[0])[:25]
        name2 = poi_name(top_trans[1])[:25]
        print(f"   Most common route: {name1} -> {name2} ({top_trans[2]} times)")

In [ ]:
# Create combined ranking
if not poi_metrics.empty:
    print("\nCOMBINED POI RANKING")
    print("="*90)
    print("POIs ranked by combined importance (popularity + bridge + influence):\n")

    # Normalize each metric to 0-1
    for col in ['popularity_score', 'bridge_score', 'influence_score']:
        max_val = poi_metrics[col].max()
        if max_val > 0:
            poi_metrics[f'{col}_norm'] = poi_metrics[col] / max_val
        else:
            poi_metrics[f'{col}_norm'] = 0

    # Combined score
    poi_metrics['combined_score'] = (
        poi_metrics['popularity_score_norm'] * 0.4 +
        poi_metrics['bridge_score_norm'] * 0.3 +
        poi_metrics['influence_score_norm'] * 0.3
    )

    print(f"{'Rank':<6} {'POI Name':<42} {'Combined':<10} {'Pop.':<8} {'Bridge':<8} {'Infl.'}")
    print("-"*90)

    for i, (_, row) in enumerate(poi_metrics.nlargest(15, 'combined_score').iterrows(), 1):
        print(f"{i:<6} {row['name']:<42} {row['combined_score']:.4f}    "
              f"{row['popularity_score']:.4f}  {row['bridge_score']:.4f}  {row['influence_score']:.4f}")

---
## Part 2 — Multi-Hop Inference Chains

Trace explicit reasoning paths through the KG connecting entities.
Each result is a chain of `(subject, predicate, object)` triples — the path *is* the explanation.

### 2.1 POI Reachability via Co-visitation Hops

POIs reachable within 1 and 2 hops of `poi:covisitedWith` (materialized by SR-05).
2-hop reach extends the recommendation surface beyond directly co-visited POIs.

In [ ]:
# 1-hop co-visitation: direct co-visited partners per POI
df_covisit_1hop = query("""
SELECT ?poi (COUNT(DISTINCT ?neighbor) AS ?reach1)
WHERE {
    ?poi poi:covisitedWith ?neighbor .
}
GROUP BY ?poi
ORDER BY DESC(?reach1)
""")

# 2-hop co-visitation: POIs reachable through an intermediate co-visited POI
df_covisit_2hop = query("""
SELECT ?poi (COUNT(DISTINCT ?hop2) AS ?reach2)
WHERE {
    ?poi poi:covisitedWith ?mid .
    ?mid poi:covisitedWith ?hop2 .
    FILTER(?hop2 != ?poi)
}
GROUP BY ?poi
ORDER BY DESC(?reach2)
""")

if not df_covisit_1hop.empty and not df_covisit_2hop.empty:
    df_covisit_1hop['reach1'] = df_covisit_1hop['reach1'].astype(int)
    df_covisit_2hop['reach2'] = df_covisit_2hop['reach2'].astype(int)
    df_reach = df_covisit_1hop.merge(df_covisit_2hop, on='poi', how='outer').fillna(0)
    df_reach['reach1'] = df_reach['reach1'].astype(int)
    df_reach['reach2'] = df_reach['reach2'].astype(int)
    df_reach['poi_name'] = df_reach['poi'].apply(short)
    df_reach = df_reach.sort_values('reach2', ascending=False)

    print(f"POIs with co-visitation links: {len(df_reach)}")
    print(f"Mean 1-hop reach: {df_reach['reach1'].mean():.1f}")
    print(f"Mean 2-hop reach: {df_reach['reach2'].mean():.1f}")
    print(f"Max 2-hop reach:  {df_reach['reach2'].max()}")
    print()
    print(df_reach[['poi_name', 'reach1', 'reach2']].head(15).to_string(index=False))
else:
    print("No co-visitation triples found. Run SR-05 materialization first.")

In [ ]:
# Reach distribution: how many POIs does each POI connect to at 1 vs 2 hops?
if not df_covisit_1hop.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(df_reach['reach1'], bins=20, edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('1-hop co-visitation reach')
    axes[0].set_ylabel('Number of POIs')
    axes[0].set_title('Direct co-visitation reach')

    axes[1].hist(df_reach['reach2'], bins=20, edgecolor='black', alpha=0.7, color='tab:orange')
    axes[1].set_xlabel('2-hop co-visitation reach')
    axes[1].set_ylabel('Number of POIs')
    axes[1].set_title('2-hop co-visitation reach')

    plt.tight_layout()
    plt.show()

**Observations — Co-visitation reachability.**
318 POIs participate in co-visitation links. Mean 1-hop reach of 1.9 means most POIs co-occur with only 1-2 others — a sparse graph dominated by a few high-connectivity nodes. 2-hop reach (mean 2.8, max 23) shows meaningful expansion: the top POI reaches 23 others at 2-hops from just 4 direct neighbors (5.75x amplification). Hub-and-spoke topology. Most POIs unreachable at 1-hop become candidates at 2 — the structural basis for multi-hop recommendation.

### 2.2 POI Reachability via Flow Chains

`poi:followedBy` (SR-06) captures sequential transitions. Transitive closure reveals
which POIs are downstream from a given starting point.

In [ ]:
# Direct flow successors per POI
df_flow_1hop = query("""
SELECT ?poi (COUNT(DISTINCT ?next) AS ?successors)
WHERE {
    ?poi poi:followedBy ?next .
}
GROUP BY ?poi
ORDER BY DESC(?successors)
""")

# 2-hop flow: POIs reachable through one intermediate step
df_flow_2hop = query("""
SELECT ?poi (COUNT(DISTINCT ?hop2) AS ?reach2)
WHERE {
    ?poi poi:followedBy ?mid .
    ?mid poi:followedBy ?hop2 .
    FILTER(?hop2 != ?poi)
}
GROUP BY ?poi
ORDER BY DESC(?reach2)
""")

if not df_flow_1hop.empty:
    df_flow_1hop['successors'] = df_flow_1hop['successors'].astype(int)
    df_flow_1hop['poi_name'] = df_flow_1hop['poi'].apply(short)
    print(f"POIs with flow-out links: {len(df_flow_1hop)}")
    print(f"Mean direct successors: {df_flow_1hop['successors'].mean():.1f}")
    print(f"Max direct successors:  {df_flow_1hop['successors'].max()}")
    print()
    print(df_flow_1hop[['poi_name', 'successors']].head(15).to_string(index=False))

    if not df_flow_2hop.empty:
        df_flow_2hop['reach2'] = df_flow_2hop['reach2'].astype(int)
        df_flow = df_flow_1hop.merge(df_flow_2hop, on='poi', how='left').fillna(0)
        df_flow['reach2'] = df_flow['reach2'].astype(int)
        expansion = df_flow['reach2'].mean() / df_flow['successors'].mean() if df_flow['successors'].mean() > 0 else 0
        print(f"\n2-hop expansion factor: {expansion:.1f}x")
else:
    print("No flow triples found. Run SR-06 materialization first.")

**Observations — Flow chains.**
Only 7 POIs have outgoing `followedBy` links — the flow graph is extremely sparse compared to co-visitation (318 POIs). This is expected: SR-06 materializes flow only between consecutive meaningful interactions via `hasNextPosition`, so only POIs visited back-to-back in the same trajectory qualify. The 0.7x 2-hop expansion (less than 1x) means 2-hop flow paths mostly duplicate existing 1-hop connections rather than opening new ones. Flow is structurally narrow: useful for sequence-level reasoning but not for broad reachability.

### 2.3 Trajectory Similarity Explanation

For any two trajectories, enumerate the shared ontological properties that make them similar.
Each shared property is a concrete triple in the graph — no opaque score.

In [ ]:
# Get a sample pair of similar trajectories (sharing purpose)
df_pairs = query("""
SELECT ?traj_a ?traj_b ?purpose
WHERE {
    ?traj_a trajecto:hasPurpose ?purpose .
    ?traj_b trajecto:hasPurpose ?purpose .
    FILTER(STR(?traj_a) < STR(?traj_b))
}
LIMIT 10
""")

if not df_pairs.empty:
    print(f"Sample trajectory pairs sharing purpose ({len(df_pairs)} shown):")
    for _, row in df_pairs.head(5).iterrows():
        print(f"  {short(row['traj_a'])} <-> {short(row['traj_b'])}  via {short(row['purpose'])}")

In [ ]:
def explain_trajectory_similarity(traj_a: str, traj_b: str) -> pd.DataFrame:
    """
    Enumerate all shared ontological properties between two trajectories.
    Returns a DataFrame of (property, sharedValue) triples.
    """
    df = query(f"""
    SELECT ?property ?sharedValue
    WHERE {{
        <{traj_a}> ?property ?sharedValue .
        <{traj_b}> ?property ?sharedValue .
        FILTER(?property IN (
            trajecto:hasPurpose,
            patterns:exhibitsPattern,
            context:hasTemporalContext,
            trajecto:hasEngagementLevel
        ))
    }}
    """)
    if not df.empty:
        df['property'] = df['property'].apply(short)
        df['sharedValue'] = df['sharedValue'].apply(short)
    return df


# Explain similarity for the first pair
if not df_pairs.empty:
    row = df_pairs.iloc[0]
    traj_a, traj_b = row['traj_a'], row['traj_b']
    print(f"Explaining similarity: {short(traj_a)} <-> {short(traj_b)}\n")

    df_explanation = explain_trajectory_similarity(traj_a, traj_b)
    if not df_explanation.empty:
        for _, r in df_explanation.iterrows():
            print(f"  {r['property']}  ->  {r['sharedValue']}")
        print(f"\nShared facets: {len(df_explanation)}")
    else:
        print("  No shared ontological properties beyond purpose.")

In [ ]:
# Distribution: how many shared facets do trajectory pairs typically have?
df_facet_counts = query("""
SELECT ?traj_a ?traj_b (COUNT(?sharedValue) AS ?sharedFacets)
WHERE {
    ?traj_a trajecto:hasPurpose ?p .
    ?traj_b trajecto:hasPurpose ?p .
    FILTER(STR(?traj_a) < STR(?traj_b))

    ?traj_a ?property ?sharedValue .
    ?traj_b ?property ?sharedValue .
    FILTER(?property IN (
        trajecto:hasPurpose,
        patterns:exhibitsPattern,
        context:hasTemporalContext,
        trajecto:hasEngagementLevel
    ))
}
GROUP BY ?traj_a ?traj_b
ORDER BY DESC(?sharedFacets)
LIMIT 500
""")

if not df_facet_counts.empty:
    df_facet_counts['sharedFacets'] = df_facet_counts['sharedFacets'].astype(int)
    counts = df_facet_counts['sharedFacets'].value_counts().sort_index()

    print("Shared facet distribution (purpose-matched pairs, sample of 500):")
    for n, c in counts.items():
        print(f"  {n} shared facets: {c} pairs")

    fig, ax = plt.subplots(figsize=(6, 4))
    counts.plot(kind='bar', ax=ax, edgecolor='black', alpha=0.7)
    ax.set_xlabel('Number of shared ontological facets')
    ax.set_ylabel('Number of trajectory pairs')
    ax.set_title('Trajectory pair similarity depth')
    plt.tight_layout()
    plt.show()

**Observations — Trajectory similarity explanation.**
The sampled pair shares only 1 facet (purpose), but the population distribution tells a different story: 408 of 500 sampled purpose-matched pairs share 3 facets, 92 share 4. This means purpose alignment is rarely isolated — it co-occurs with temporal context and engagement level matches. The high co-occurrence of 3+ shared facets means that purpose-based blocking (used in SR-11..13) is a strong proxy for multi-dimensional similarity. Pairs sharing 4 facets (including navigation pattern) are the highest-confidence matches.

### 2.4 User-to-POI Inference Chains

Trace how a user connects to a POI they never visited, through purpose-matched trajectories
and co-visitation links. Each step is an explicit triple.

In [ ]:
# For a sample user, find POIs reachable through:
# User -> Trajectory_A -> purpose -> Trajectory_B -> POI_X -> covisitedWith -> POI_Y

df_chains = query("""
SELECT ?user ?traj_a ?purpose ?traj_b ?poi_x ?poi_y ?category
WHERE {
    ?user trajecto:formsTrajectory ?traj_a .
    ?traj_a trajecto:hasPurpose ?purpose .

    ?traj_b trajecto:hasPurpose ?purpose .
    FILTER(?traj_b != ?traj_a)

    # Ensure traj_b is NOT from the same user
    ?other_user trajecto:formsTrajectory ?traj_b .
    FILTER(?other_user != ?user)

    ?pos dul:isPartOf ?traj_b .
    ?pos trajecto:hasInteraction ?int .
    ?int a poi:MeaningfulInteraction .
    ?int poi:interactsWith ?poi_x .

    ?poi_x poi:covisitedWith ?poi_y .

    # POI_Y must not appear in user's own trajectories
    FILTER NOT EXISTS {
        ?user trajecto:formsTrajectory ?own_traj .
        ?own_pos dul:isPartOf ?own_traj .
        ?own_pos trajecto:hasInteraction ?own_int .
        ?own_int poi:interactsWith ?poi_y .
    }

    ?poi_y a ?category .
    FILTER(?category != trajecto:PointOfInterest && ?category != poi:SignificantPOI)
}
LIMIT 30
""")

if not df_chains.empty:
    print(f"User-to-unseen-POI inference chains: {len(df_chains)}\n")
    for _, r in df_chains.head(10).iterrows():
        print(f"  {short(r['user'])}")
        print(f"    -> traj {short(r['traj_a'])}")
        print(f"    -> purpose {short(r['purpose'])}")
        print(f"    -> traj {short(r['traj_b'])} (other user)")
        print(f"    -> {short(r['poi_x'])}")
        print(f"    -> covisitedWith -> {short(r['poi_y'])} [{short(r['category'])}]")
        print()
else:
    print("No inference chains found. Ensure SR-05 (co-visitation) is materialized.")

**Observations — User-to-POI inference chains.**
The 4-hop chains successfully surface unseen POIs through purpose-matched trajectories. Each chain is fully traceable through explicit triples.

Data quality note: one chain recommends `TransportationPOI` — these should be filtered via `EXCLUDED_CATEGORIES` in production. The chains also expose cross-category reach: a Cultural purpose chain surfaces Monuments, Museums, and Restaurants (co-visited with cultural POIs). This cross-category bleed is a feature, not a bug — it captures real visit behavior (cultural visitors eat nearby).

### 2.5 POI Relatedness Chains

POIs connected through mixed flow (`followedBy`) and co-visitation (`covisitedWith`) paths,
with category coherence check.

In [ ]:
# POIs connected: A -[followedBy]-> B -[covisitedWith]-> C, with category info
df_poi_chains = query("""
SELECT ?poi_a ?cat_a ?poi_b ?poi_c ?cat_c
WHERE {
    ?poi_a poi:followedBy ?poi_b .
    ?poi_b poi:covisitedWith ?poi_c .
    FILTER(?poi_c != ?poi_a)

    ?poi_a a ?cat_a .
    FILTER(?cat_a != trajecto:PointOfInterest && ?cat_a != poi:SignificantPOI)

    ?poi_c a ?cat_c .
    FILTER(?cat_c != trajecto:PointOfInterest && ?cat_c != poi:SignificantPOI)
}
LIMIT 200
""")

if not df_poi_chains.empty:
    df_poi_chains['cat_a_name'] = df_poi_chains['cat_a'].apply(short)
    df_poi_chains['cat_c_name'] = df_poi_chains['cat_c'].apply(short)
    df_poi_chains['same_category'] = df_poi_chains['cat_a'] == df_poi_chains['cat_c']

    n_same = df_poi_chains['same_category'].sum()
    n_total = len(df_poi_chains)
    print(f"Mixed flow+covisit chains: {n_total}")
    print(f"Category-coherent (A and C same category): {n_same} ({100*n_same/n_total:.1f}%)\n")

    # Category pair distribution
    pair_counts = df_poi_chains.groupby(['cat_a_name', 'cat_c_name']).size().reset_index(name='count')
    pair_counts = pair_counts.sort_values('count', ascending=False)
    print("Top category pairs (A -> B -> C):")
    print(pair_counts.head(15).to_string(index=False))
else:
    print("No mixed chains found. Ensure SR-05 and SR-06 are materialized.")

**Observations — POI relatedness chains.**
44 mixed flow+covisit chains, but only 11.4% are category-coherent (A and C share a category). BusStation and Restaurant dominate as intermediate hops — they are the most-connected POIs in both graphs, acting as structural bridges between POI clusters regardless of category.

The `None` category appearing in results indicates POIs without an assigned ontology class — a data quality gap. The low category coherence (11.4%) suggests that mixed-predicate chains are better suited for discovery (surfacing unexpected connections) than for category-constrained recommendations.

### 2.6 Path Length Distribution

How many hops separate users from recommended POIs? Shorter paths = stronger evidence.

In [ ]:
# Count POIs reachable at each hop distance via co-visitation
hop_stats = {}

# 1-hop
df_h1 = query("""
SELECT (COUNT(DISTINCT ?pair) AS ?cnt)
WHERE {
    ?a poi:covisitedWith ?b .
    BIND(CONCAT(STR(?a), "-", STR(?b)) AS ?pair)
}
""")
hop_stats[1] = int(df_h1.iloc[0]['cnt']) if not df_h1.empty else 0

# 2-hop (new connections not in 1-hop)
df_h2 = query("""
SELECT (COUNT(DISTINCT ?pair) AS ?cnt)
WHERE {
    ?a poi:covisitedWith ?mid .
    ?mid poi:covisitedWith ?b .
    FILTER(?a != ?b)
    FILTER NOT EXISTS { ?a poi:covisitedWith ?b }
    BIND(CONCAT(STR(?a), "-", STR(?b)) AS ?pair)
}
""")
hop_stats[2] = int(df_h2.iloc[0]['cnt']) if not df_h2.empty else 0

print("Co-visitation reachability by hop distance:")
for hop, cnt in hop_stats.items():
    print(f"  {hop}-hop: {cnt} unique POI pairs")

if hop_stats[1] > 0:
    print(f"\n2-hop adds {hop_stats[2]} new connections beyond 1-hop ({hop_stats[1]})")
    print(f"Expansion: {(hop_stats[1] + hop_stats[2]) / hop_stats[1]:.1f}x total reach")

**Observations — Path length distribution.**
592 unique POI pairs at 1-hop, 865 new pairs at 2-hop — a 2.5x total reach expansion. This quantifies the recommendation surface gain from multi-hop inference: without 2-hop traversal, 59% of reachable POI pairs would be invisible. For the recommendation engine, this means 2-hop co-visitation is not optional but structurally necessary to achieve adequate coverage.

---
## Part 3 — Ontological Cross-Tabulations

Cross-tabulations over OWL class-valued properties.
Subsumption is declared in the ontology; queries respect it without explicit enumeration.

### 3.1 Interaction Type Distribution per Purpose

Do Cultural trajectories have proportionally more immersive interactions than Dining?

In [ ]:
df_purpose_int = query("""
SELECT ?purpose ?interactionType (COUNT(?int) AS ?count)
WHERE {
    ?traj trajecto:hasPurpose ?purpose .
    ?pos dul:isPartOf ?traj .
    ?pos trajecto:hasInteraction ?int .
    ?int a ?interactionType .
    FILTER(?interactionType IN (
        poi:PassingInteraction,
        poi:MeaningfulInteraction,
        poi:ImmersiveInteraction
    ))
}
GROUP BY ?purpose ?interactionType
ORDER BY ?purpose ?interactionType
""")

if not df_purpose_int.empty:
    df_purpose_int['purpose'] = df_purpose_int['purpose'].apply(short)
    df_purpose_int['interactionType'] = df_purpose_int['interactionType'].apply(short)
    df_purpose_int['count'] = df_purpose_int['count'].astype(int)

    pivot = df_purpose_int.pivot_table(
        index='purpose', columns='interactionType', values='count', fill_value=0
    )
    # Reorder columns by hierarchy
    col_order = [c for c in ['PassingInteraction', 'MeaningfulInteraction', 'ImmersiveInteraction'] if c in pivot.columns]
    pivot = pivot[col_order]

    print("Interaction counts per purpose:")
    print(pivot)

    # Proportional view
    pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
    print("\nProportional (% of interactions per purpose):")
    print(pivot_pct.round(1))
else:
    print("No results. Check that trajectories have hasPurpose and interactions have types.")

In [ ]:
# Heatmap of proportional interaction types
if not df_purpose_int.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sns.heatmap(pivot, annot=True, fmt='g', cmap='Blues', ax=axes[0])
    axes[0].set_title('Interaction counts per purpose')
    axes[0].set_ylabel('')

    sns.heatmap(pivot_pct, annot=True, fmt='.1f', cmap='YlOrRd', ax=axes[1])
    axes[1].set_title('Interaction distribution % per purpose')
    axes[1].set_ylabel('')

    plt.tight_layout()
    plt.show()

**Observations — Purpose x Interaction type.**
CommercialPurpose dominates raw counts, reflecting the dataset's urban commercial landscape. CulturalPurpose has the highest immersive interaction share proportionally.

BusStationTransit shows 25 immersive interactions — all from a single POI with 192s dwell time. A 3-minute wait at a bus stop misclassified as deep engagement. TransportationPurpose has 60 immersive — same artifact from dwell-time-based classification applied to transit contexts where waiting ≠ engagement. These transit purposes should be excluded from immersive analysis or the classification should incorporate POI category as a gate.

### 3.2 Navigation Style x Temporal Context

Is exploratory movement concentrated in mornings? Are targeted segments more common on weekdays?

In [ ]:
df_nav_temporal = query("""
SELECT ?navStyle ?temporal (COUNT(DISTINCT ?traj) AS ?count)
WHERE {
    ?traj patterns:exhibitsPattern ?navStyle .
    ?traj context:hasTemporalContext ?temporal .
}
GROUP BY ?navStyle ?temporal
ORDER BY ?navStyle DESC(?count)
""")

if not df_nav_temporal.empty:
    df_nav_temporal['navStyle'] = df_nav_temporal['navStyle'].apply(short)
    df_nav_temporal['temporal'] = df_nav_temporal['temporal'].apply(short)
    df_nav_temporal['count'] = df_nav_temporal['count'].astype(int)

    pivot_nav = df_nav_temporal.pivot_table(
        index='navStyle', columns='temporal', values='count', fill_value=0
    )

    print("Navigation style x Temporal context (trajectory count):")
    print(pivot_nav)

    fig, ax = plt.subplots(figsize=(12, 5))
    sns.heatmap(pivot_nav, annot=True, fmt='g', cmap='YlGnBu', ax=ax)
    ax.set_title('Navigation style x Temporal context')
    ax.set_ylabel('')
    plt.tight_layout()
    plt.show()
else:
    print("No results. Check patterns:exhibitsPattern and hasTemporalContext triples.")

**Observations — Navigation x Temporal.**
Previous run returned empty because the query used `trajecto:hasTemporalContext` — the actual predicate is `context:hasTemporalContext` (context namespace, not trajecto). Fixed above.

### 3.3 Engagement Level x POI Category

Which POI categories attract high-engagement trajectories?

In [ ]:
df_eng_cat = query("""
SELECT ?category ?engLevel (COUNT(DISTINCT ?traj) AS ?count)
WHERE {
    ?traj trajecto:hasEngagementLevel ?engLevel .
    ?pos dul:isPartOf ?traj .
    ?pos trajecto:hasInteraction ?int .
    ?int a poi:MeaningfulInteraction .
    ?int poi:interactsWith ?poi .
    ?poi a ?category .
    FILTER(?category != trajecto:PointOfInterest && ?category != poi:SignificantPOI)
}
GROUP BY ?category ?engLevel
ORDER BY ?category ?engLevel
""")

if not df_eng_cat.empty:
    df_eng_cat['category'] = df_eng_cat['category'].apply(short)
    df_eng_cat['engLevel'] = df_eng_cat['engLevel'].apply(short)
    df_eng_cat['count'] = df_eng_cat['count'].astype(int)

    pivot_eng = df_eng_cat.pivot_table(
        index='category', columns='engLevel', values='count', fill_value=0
    )

    # Sort by total engagement
    pivot_eng['_total'] = pivot_eng.sum(axis=1)
    pivot_eng = pivot_eng.sort_values('_total', ascending=False).drop('_total', axis=1)

    print(f"Engagement level x POI category ({len(pivot_eng)} categories):")
    print(pivot_eng.head(20))

    fig, ax = plt.subplots(figsize=(10, max(6, len(pivot_eng) * 0.35)))
    sns.heatmap(pivot_eng.head(20), annot=True, fmt='g', cmap='RdYlGn', ax=ax)
    ax.set_title('Engagement level x POI category (top 20)')
    ax.set_ylabel('')
    plt.tight_layout()
    plt.show()
else:
    print("No results. Ensure SR-10 (engagement levels) is materialized.")

**Observations — Engagement x POI category.**
LowEngagement dominates across all categories. ArtworkPOI stands out: 12 MediumEngagement vs 11 LowEngagement — proportionally the highest engagement category. Monument has a 15% high-engagement rate.

`None` category = 17 POIs that were never assigned a specific type by the GPT classifier. The top unclassified POI is "Il Cuore di Nerone" with 230 meaningful interactions — a significant data gap. These POIs have only the generic `PointOfInterest` type. They should be manually classified or re-run through the classifier.

### 3.4 Purpose Distribution per User

Purpose consistency vs. diversity across users.

In [ ]:
df_user_purpose = query("""
SELECT ?user ?purpose (COUNT(DISTINCT ?traj) AS ?trajectoryCount)
WHERE {
    ?user trajecto:formsTrajectory ?traj .
    ?traj trajecto:hasPurpose ?purpose .
}
GROUP BY ?user ?purpose
ORDER BY ?user DESC(?trajectoryCount)
""")

if not df_user_purpose.empty:
    df_user_purpose['user'] = df_user_purpose['user'].apply(short)
    df_user_purpose['purpose'] = df_user_purpose['purpose'].apply(short)
    df_user_purpose['trajectoryCount'] = df_user_purpose['trajectoryCount'].astype(int)

    # Per-user purpose entropy
    user_stats = []
    for user, group in df_user_purpose.groupby('user'):
        counts = group['trajectoryCount'].values
        total = counts.sum()
        probs = counts / total
        entropy = -np.sum(probs * np.log2(probs + 1e-10))
        dominant = group.loc[group['trajectoryCount'].idxmax(), 'purpose']
        user_stats.append({
            'user': user,
            'total_trajectories': total,
            'unique_purposes': len(group),
            'purpose_entropy': entropy,
            'dominant_purpose': dominant
        })

    df_ustats = pd.DataFrame(user_stats).sort_values('purpose_entropy', ascending=False)

    print("User purpose profiles:")
    print(df_ustats.to_string(index=False))

    if len(df_ustats) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        axes[0].bar(df_ustats['user'], df_ustats['purpose_entropy'], edgecolor='black', alpha=0.7)
        axes[0].set_xlabel('User')
        axes[0].set_ylabel('Purpose entropy (bits)')
        axes[0].set_title('Purpose diversity per user')
        axes[0].tick_params(axis='x', rotation=45)

        purpose_counts = df_user_purpose.groupby('purpose')['trajectoryCount'].sum().sort_values(ascending=False)
        axes[1].bar(purpose_counts.index, purpose_counts.values, edgecolor='black', alpha=0.7, color='tab:orange')
        axes[1].set_xlabel('Purpose')
        axes[1].set_ylabel('Total trajectories')
        axes[1].set_title('Purpose frequency across all users')
        axes[1].tick_params(axis='x', rotation=45)

        plt.tight_layout()
        plt.show()
else:
    print("No results.")

**Observations — User purpose profiles.**
Giardia dominates with 332 trajectories across 10 purposes (entropy 1.97) — a power user with diverse behavior. Most users have a single trajectory (entropy ~0) — insufficient data for meaningful purpose profiling.

CommercialPurpose is the dominant purpose for the majority of users, consistent with the urban dataset. Only 3 users (Giardia, richlv, 6arms1leg) have enough trajectories and purpose diversity for reliable preference learning.

The negative entropy values (-1.44e-10) for single-purpose users are a floating-point artifact of `log2(1.0 + epsilon)`. Harmless but should be clamped to 0 for display.

### 3.5 Purpose x Navigation Style

Do certain purposes correlate with specific movement behaviors?

In [ ]:
df_purpose_nav = query("""
SELECT ?purpose ?navStyle (COUNT(DISTINCT ?traj) AS ?count)
WHERE {
    ?traj trajecto:hasPurpose ?purpose .
    ?traj patterns:exhibitsPattern ?navStyle .
}
GROUP BY ?purpose ?navStyle
ORDER BY ?purpose ?navStyle
""")

if not df_purpose_nav.empty:
    df_purpose_nav['purpose'] = df_purpose_nav['purpose'].apply(short)
    df_purpose_nav['navStyle'] = df_purpose_nav['navStyle'].apply(short)
    df_purpose_nav['count'] = df_purpose_nav['count'].astype(int)

    pivot_pn = df_purpose_nav.pivot_table(
        index='purpose', columns='navStyle', values='count', fill_value=0
    )

    print("Purpose x Navigation style:")
    print(pivot_pn)

    # Proportional
    pivot_pn_pct = pivot_pn.div(pivot_pn.sum(axis=1), axis=0) * 100
    print("\nProportional (% per purpose):")
    print(pivot_pn_pct.round(1))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.heatmap(pivot_pn, annot=True, fmt='g', cmap='Blues', ax=axes[0])
    axes[0].set_title('Purpose x Navigation (counts)')
    axes[0].set_ylabel('')

    sns.heatmap(pivot_pn_pct, annot=True, fmt='.1f', cmap='YlOrRd', ax=axes[1])
    axes[1].set_title('Purpose x Navigation (% per purpose)')
    axes[1].set_ylabel('')

    plt.tight_layout()
    plt.show()

**Observations — Purpose x Navigation style.**
TargetedSegment dominates almost every purpose (67-100%), consistent with urban GPS traces where users move between specific POIs rather than wandering.

CulturalPurpose has the highest ExploratoryMovement share (10%) — cultural visitors explore neighborhoods, not just individual sites. RecreationalPurpose is 100% CasualMovement (0% Targeted) — pure leisure movement without specific destination targeting.

ImmersiveExploration does not appear for any purpose. Either the classification threshold is too high, or the pattern is not being assigned by the pipeline.

The fractional CasualMovement values (1.0, 1.5, 2.0) suggest a trajectory can have multiple navigation patterns assigned, and the COUNT(DISTINCT ?traj) is counting trajectory-purpose-pattern combinations.

### 3.6 Cross-Cutting: Temporal x Interaction Type x Purpose

Three-way aggregation. A single SPARQL query producing a result
that would require 3-table joins in a relational model.

In [ ]:
df_3way = query("""
SELECT ?temporal ?purpose ?interactionType (COUNT(?int) AS ?count)
WHERE {
    ?traj context:hasTemporalContext ?temporal .
    ?traj trajecto:hasPurpose ?purpose .
    ?pos dul:isPartOf ?traj .
    ?pos trajecto:hasInteraction ?int .
    ?int a ?interactionType .
    FILTER(?interactionType IN (
        poi:PassingInteraction,
        poi:MeaningfulInteraction,
        poi:ImmersiveInteraction
    ))
    # Focus on time-of-day contexts for readability
    FILTER(?temporal IN (
        context:Morning, context:Afternoon, context:Evening, context:Night
    ))
}
GROUP BY ?temporal ?purpose ?interactionType
ORDER BY ?temporal ?purpose ?interactionType
""")

if not df_3way.empty:
    df_3way['temporal'] = df_3way['temporal'].apply(short)
    df_3way['purpose'] = df_3way['purpose'].apply(short)
    df_3way['interactionType'] = df_3way['interactionType'].apply(short)
    df_3way['count'] = df_3way['count'].astype(int)

    # Show as one heatmap per temporal context
    temporals = df_3way['temporal'].unique()
    n_temps = len(temporals)

    if n_temps > 0:
        fig, axes = plt.subplots(1, n_temps, figsize=(5 * n_temps, 5), squeeze=False)
        for i, temp in enumerate(sorted(temporals)):
            subset = df_3way[df_3way['temporal'] == temp]
            piv = subset.pivot_table(
                index='purpose', columns='interactionType', values='count', fill_value=0
            )
            col_order = [c for c in ['PassingInteraction', 'MeaningfulInteraction', 'ImmersiveInteraction'] if c in piv.columns]
            piv = piv[col_order] if col_order else piv
            sns.heatmap(piv, annot=True, fmt='g', cmap='Blues', ax=axes[0][i])
            axes[0][i].set_title(f'{temp}')
            axes[0][i].set_ylabel('' if i > 0 else 'Purpose')

        plt.suptitle('Purpose x Interaction Type by Time of Day', y=1.02)
        plt.tight_layout()
        plt.show()
else:
    print("No 3-way results.")

### 3.7 Statistical Association Tests

Chi-squared test on the contingency tables to identify statistically significant associations
between class-valued properties.

In [ ]:
from scipy.stats import chi2_contingency

def test_association(pivot_table: pd.DataFrame, name: str):
    """Run chi-squared test on a contingency table."""
    if pivot_table.empty or pivot_table.shape[0] < 2 or pivot_table.shape[1] < 2:
        print(f"{name}: insufficient dimensions for chi-squared test.")
        return None

    chi2, p, dof, expected = chi2_contingency(pivot_table.values)
    # Cramer's V for effect size
    n = pivot_table.values.sum()
    k = min(pivot_table.shape) - 1
    cramers_v = np.sqrt(chi2 / (n * k)) if n * k > 0 else 0

    print(f"{name}:")
    print(f"  chi2 = {chi2:.2f}, dof = {dof}, p = {p:.2e}")
    print(f"  Cramer's V = {cramers_v:.3f}")
    sig = 'significant' if p < 0.05 else 'not significant'
    strength = 'weak' if cramers_v < 0.1 else 'moderate' if cramers_v < 0.3 else 'strong'
    print(f"  Association: {sig} (p<0.05), {strength} effect")
    print()
    return {'chi2': chi2, 'p': p, 'dof': dof, 'cramers_v': cramers_v}

results = {}

# Test each contingency table produced above
if 'pivot' in dir() and not pivot.empty:
    results['Purpose x Interaction'] = test_association(pivot, 'Purpose x Interaction Type')

if 'pivot_nav' in dir() and not pivot_nav.empty:
    results['Navigation x Temporal'] = test_association(pivot_nav, 'Navigation Style x Temporal Context')

if 'pivot_pn' in dir() and not pivot_pn.empty:
    results['Purpose x Navigation'] = test_association(pivot_pn, 'Purpose x Navigation Style')

if 'pivot_eng' in dir() and not pivot_eng.empty:
    results['Engagement x Category'] = test_association(pivot_eng, 'Engagement Level x POI Category')

In [ ]:
# Summary of associations
if results:
    summary = []
    for name, r in results.items():
        if r:
            summary.append({
                'Association': name,
                'chi2': f"{r['chi2']:.1f}",
                'p-value': f"{r['p']:.2e}",
                "Cramer's V": f"{r['cramers_v']:.3f}",
                'Significant': r['p'] < 0.05
            })

    df_summary = pd.DataFrame(summary)
    print("Association test summary:")
    print(df_summary.to_string(index=False))

**Observations — Statistical associations.**
All three tested associations are statistically significant (p < 0.05):

- **Engagement x Category** has the strongest effect (Cramer's V = 0.375, strong). POI category is a meaningful predictor of engagement level — not surprising given that cultural/artwork POIs consistently attract deeper engagement than commercial ones.
- **Purpose x Navigation** (V = 0.220, moderate). Purpose constrains movement style — recreational visitors move casually, cultural visitors explore, commercial visitors target.
- **Purpose x Interaction** (V = 0.190, moderate). The weakest of the three, suggesting that interaction type is more determined by the specific POI encountered than by the trajectory's overall purpose.

Navigation x Temporal could not be tested (no data). This is the most theoretically interesting association — whether time-of-day affects movement behavior — and should be prioritized after fixing the temporal context query.